# 11.5 — Monte Carlo Methods

Monte Carlo methods learn from complete sampled episodes: wait until the trajectory ends, turn the realized rewards into discounted returns, and average those returns into value or action-value estimates. In this lesson, every algorithm is built from small NumPy tables on tiny environments so the distinction between reward, return, first-visit/every-visit averaging, control, and importance sampling stays visible.

## 📖 Concept walkthrough — build each idea from scratch

Before the terse worked examples, we build Monte Carlo methods one idea at a time. Run each cell in order and read the printed intermediate values — every piece of math is spelled out so nothing is a black box. This walkthrough is self-contained (it imports what it needs) and uses a `_w` suffix on its variables so it never clashes with the examples below.

In [ ]:
import numpy as np  # arrays, random sampling, and table updates for Monte Carlo RL.
import matplotlib.pyplot as plt  # small visualizations for returns, values, and policies.
np.random.seed(0)  # reproducibility for sampled episodes.

### 1. Episodes and discounted returns

Monte Carlo learning starts only after an **episode** finishes. A reward is one immediate number, but a return is the discounted ledger of everything that happened afterward: $G_t=R_{t+1}+\gamma R_{t+2}+\gamma^2R_{t+3}+\cdots$. The discount $\gamma$ makes delayed rewards count, but not as strongly as rewards received now.

In [ ]:
rewards_w = np.array([1.0, 0.0, 2.0])  # rewards collected after three successive decisions.
gamma_w = 0.9  # discount factor: one step later is worth 90% as much as now.
powers_w = gamma_w ** np.arange(len(rewards_w))  # [1, gamma, gamma^2] for a return from time 0.

print("rewards:", rewards_w)  # inspect immediate rewards before summing them.
print("discount powers:", np.round(powers_w, 3))  # inspect how the future is down-weighted.

▶ What you'll see: the third reward is multiplied by `0.9²`, so it still matters but is discounted.

In [ ]:
G0_w = float(np.sum(powers_w * rewards_w))  # compute 1 + 0.9*0 + 0.9^2*2.

print("G0:", round(G0_w, 3))  # inspect the three-step return.

assert round(G0_w, 3) == 2.62  # concrete check from the lesson grounding.

▶ What you'll see: `G0 = 2.620`, not just the immediate reward `1.000`.

In [ ]:
returns_w = []  # store G_t for each time step in the episode.
for t_w in range(len(rewards_w)):  # compute a return starting from each time index.
    discounts_w = gamma_w ** np.arange(len(rewards_w) - t_w)  # powers for the remaining suffix.
    returns_w.append(float(np.sum(discounts_w * rewards_w[t_w:])))  # discounted suffix sum.
returns_w = np.array(returns_w)  # convert to an array for plotting.

print("returns G_t:", np.round(returns_w, 3))  # inspect all Monte Carlo targets in the episode.

plt.figure(figsize=(4.5, 3))
plt.bar(["G0", "G1", "G2"], returns_w, color="teal")
plt.title("1: each state gets a suffix return")
plt.ylabel("discounted return")
plt.show()

▶ What you'll see: later time steps have shorter ledgers, so `G2` is just the final reward.

*Why it's done this way:* Monte Carlo does not bootstrap from a guessed next value; it waits for the real suffix of rewards. The formula is a discounted sum because the target for a state is not "what happened immediately?" but "what consequence followed this visit under the policy?"

### 2. Monte Carlo prediction: first-visit and every-visit averages

To estimate $V^\pi(s)$, run episodes under a fixed policy, compute returns, and average the returns observed after visits to state $s$. **First-visit MC** uses only the first time a state appears in an episode; **every-visit MC** uses every occurrence. Both are sample averages of complete returns, but they count evidence differently when a state repeats.

In [ ]:
episode_w = [(0, 0), (1, 0), (0, 1), (2, 0)]  # (state, reward-after-that-state) pairs; state 0 repeats.
states_w = np.array([x[0] for x in episode_w])  # state sequence in the episode.
rewards2_w = np.array([x[1] for x in episode_w], dtype=float)  # reward after each state visit.

print("states:", states_w)  # inspect visits; state 0 appears at times 0 and 2.
print("rewards:", rewards2_w)  # inspect rewards used to build returns.

▶ What you'll see: state `0` repeats, which is exactly where first-visit and every-visit differ.

In [ ]:
gamma2_w = 0.9  # discount for this episode.
G_w = np.zeros_like(rewards2_w)  # allocate one return per time step.
running_w = 0.0  # suffix return built backward.
for t_w in range(len(rewards2_w) - 1, -1, -1):  # walk backward through the episode.
    running_w = rewards2_w[t_w] + gamma2_w * running_w  # G_t = R_{t+1} + gamma G_{t+1}.
    G_w[t_w] = running_w  # save the return after this visit.

print("returns:", np.round(G_w, 3))  # inspect Monte Carlo targets for each visit.

assert np.allclose(np.round(G_w, 3), [0.81, 0.9, 1.0, 0.0])  # verify the backward ledger.

▶ What you'll see: the early state-0 visit gets `0.81`, while the later state-0 visit gets `1.0`.

In [ ]:
first_visits_w = []  # returns used by first-visit MC for state 0.
every_visits_w = []  # returns used by every-visit MC for state 0.
seen_w = set()  # states already counted once in this episode.
for t_w, s_w in enumerate(states_w):  # scan episode from start to finish.
    if s_w == 0:  # collect evidence for state 0.
        every_visits_w.append(G_w[t_w])  # every-visit MC always counts it.
        if s_w not in seen_w:  # first-visit MC counts only the first occurrence.
            first_visits_w.append(G_w[t_w])
    seen_w.add(s_w)  # mark the state after processing this time step.

print("first-visit state 0 estimate:", round(float(np.mean(first_visits_w)), 3))
print("every-visit state 0 estimate:", round(float(np.mean(every_visits_w)), 3))

▶ What you'll see: first-visit uses `0.81`; every-visit averages `0.81` and `1.0` into `0.905`.

In [ ]:
plt.figure(figsize=(4.5, 3))
plt.bar(["first-visit", "every-visit"], [np.mean(first_visits_w), np.mean(every_visits_w)], color=["teal", "purple"])
plt.scatter([0], first_visits_w, color="black", zorder=3, label="counted return")
plt.scatter(np.ones(len(every_visits_w)), every_visits_w, color="black", zorder=3)
plt.title("2: repeated visits change the average")
plt.ylabel("state-0 value estimate")
plt.legend()
plt.show()

▶ What you'll see: every-visit uses both state-0 returns, so its bar sits between the two black samples.

*Why it's done this way:* $V^\pi(s)$ is an expectation over returns after visiting $s$. A sample average estimates that expectation; the first/every-visit choice decides whether repeated visits inside one episode contribute once or multiple times, which changes variance and correlation but not the basic target.

### 3. Incremental averaging and variance

Monte Carlo estimates are averages over noisy episode returns. Instead of storing every return forever, we update a running mean with $V_n=V_{n-1}+\frac{1}{n}(G_n-V_{n-1})$. The difference $G_n-V_{n-1}$ is the new sample's surprise, and $1/n$ gives the new sample exactly one share of the average.

In [ ]:
samples_w = np.array([2.0, 4.0, 1.0, 3.0, 5.0])  # five returns sampled for the same state.
mean_w = 0.0  # initial estimate before seeing samples.
means_w = []  # store the running estimate after each sample.
for n_w, g_w in enumerate(samples_w, start=1):
    mean_w = mean_w + (g_w - mean_w) / n_w  # incremental sample mean update.
    means_w.append(mean_w)  # save for inspection.

print("running means:", np.round(means_w, 3))

assert round(means_w[-1], 3) == 3.0  # equals the ordinary average of the five returns.

▶ What you'll see: the running estimate moves a lot early, then less as more evidence accumulates.

In [ ]:
ordinary_mean_w = float(np.mean(samples_w))  # compute the same average directly.
incremental_mean_w = float(means_w[-1])  # final running estimate.

print("ordinary mean:", ordinary_mean_w, "incremental mean:", incremental_mean_w)

plt.figure(figsize=(4.5, 3))
plt.plot(np.arange(1, len(samples_w) + 1), means_w, marker="o", color="purple")
plt.axhline(ordinary_mean_w, color="black", linestyle="--", label="final average")
plt.title("3: Monte Carlo sample average")
plt.xlabel("number of returns")
plt.ylabel("V estimate")
plt.legend()
plt.show()

▶ What you'll see: the curve converges to the same mean you would get by storing all returns.

In [ ]:
rng_w = np.random.default_rng(0)  # local randomness for a variance demo.
rolls_w = rng_w.normal(loc=2.0, scale=3.0, size=200)  # noisy returns with true mean near 2.
avg_curve_w = np.cumsum(rolls_w) / np.arange(1, len(rolls_w) + 1)  # running MC estimates.

print("first estimate:", round(float(avg_curve_w[0]), 3), "last estimate:", round(float(avg_curve_w[-1]), 3))

plt.figure(figsize=(5, 3))
plt.plot(avg_curve_w, color="seagreen")
plt.axhline(2.0, color="black", linestyle="--", label="true mean")
plt.title("3: noisy returns average out slowly")
plt.xlabel("episode")
plt.ylabel("estimate")
plt.legend()
plt.show()

▶ What you'll see: high-return variance makes the estimate wobble before settling near the true mean.

*Why it's done this way:* complete returns are unbiased targets for the policy value, but they can be high variance because every random future event is included. Incremental averaging keeps the correct sample-mean target while making memory constant and visibly reducing noise across episodes.

### 4. Monte Carlo control with action values and exploration

Prediction evaluates a fixed policy; control improves a policy. We estimate $Q(s,a)$ from returns after state-action pairs, then make the policy greedier with respect to $Q$. Exploration is required because a purely greedy agent might never sample the action that would reveal a better return.

In [ ]:
Q_w = np.zeros((3, 2))  # three states, two actions; action-value table starts unknown.
N_w = np.zeros_like(Q_w)  # visit counts for incremental averages.
episode3_w = [(0, 1, 0.0), (1, 0, 0.0), (2, 1, 3.0)]  # (state, action, reward).
gamma3_w = 0.9  # discount for action returns.

print("Q shape:", Q_w.shape)  # inspect |S| x |A| object shape.
print("episode:", episode3_w)  # inspect the complete trajectory before updating.

▶ What you'll see: action values are a table, not a single scalar value per state.

In [ ]:
G3_w = 0.0  # suffix return accumulator.
returns3_w = []  # returns paired with each state-action visit.
for s_w, a_w, r_w in reversed(episode3_w):  # compute returns backward.
    G3_w = r_w + gamma3_w * G3_w  # discounted consequence from this state-action pair.
    returns3_w.append((s_w, a_w, G3_w))  # store the target.
returns3_w = list(reversed(returns3_w))  # restore chronological order.

print("state-action returns:", [(s, a, round(g, 3)) for s, a, g in returns3_w])

▶ What you'll see: the earliest action receives discounted credit for the final reward.

In [ ]:
for s_w, a_w, g_w in returns3_w:  # update each visited state-action pair once.
    N_w[s_w, a_w] += 1  # count this sample.
    Q_w[s_w, a_w] += (g_w - Q_w[s_w, a_w]) / N_w[s_w, a_w]  # sample-average update.
policy_w = np.argmax(Q_w, axis=1)  # greedy improvement: choose highest estimated action per state.

print("Q after one episode:\n", np.round(Q_w, 3))
print("greedy actions:", policy_w)

plt.figure(figsize=(4.5, 3))
plt.imshow(Q_w, cmap="viridis", aspect="auto")
plt.colorbar(label="Q(s,a)")
plt.title("4: action values after MC return updates")
plt.xlabel("action")
plt.ylabel("state")
plt.show()

▶ What you'll see: only visited state-action cells changed; greedy actions follow the larger row entries.

*Why it's done this way:* control needs action values because policy improvement must compare actions in the same state. Exploration keeps all relevant actions sampled, while the greedy step converts estimated consequences into better behavior.

### 5. Off-policy evaluation and importance sampling

Sometimes the data comes from a behavior policy $b$ while we want to evaluate a target policy $\pi$. Importance sampling reweights a sampled return by the likelihood ratio $\rho=\prod_t \pi(A_t|S_t)/b(A_t|S_t)$. If the behavior took actions the target likes, the episode counts more; if it took actions the target would never take, the episode counts zero.

In [ ]:
pi_probs_w = np.array([0.8, 0.2])  # target policy probabilities for two sampled actions.
b_probs_w = np.array([0.5, 0.5])  # behavior policy probabilities for those same actions.
ratios_w = pi_probs_w / b_probs_w  # per-step likelihood ratios.
rho_w = float(np.prod(ratios_w))  # trajectory-level importance weight.

print("per-step ratios:", ratios_w)
print("trajectory weight rho:", round(rho_w, 3))

assert round(rho_w, 3) == 0.64  # (0.8/0.5)*(0.2/0.5).

▶ What you'll see: the episode is down-weighted overall because the target policy dislikes the second action.

In [ ]:
G_off_w = 3.0  # return observed under the behavior policy.
weighted_return_w = rho_w * G_off_w  # ordinary importance-sampling contribution.

print("raw return:", G_off_w, "weighted return:", round(weighted_return_w, 3))

▶ What you'll see: the return `3.0` contributes only `1.92` after policy correction.

In [ ]:
weights_w = np.array([0.64, 1.20, 0.00, 2.10])  # four episode likelihood ratios.
returns_off_w = np.array([3.0, 1.0, 5.0, 2.0])  # four behavior-policy returns.
ordinary_w = float(np.mean(weights_w * returns_off_w))  # ordinary IS average.
weighted_w = float(np.sum(weights_w * returns_off_w) / np.sum(weights_w))  # weighted IS estimate.

print("ordinary IS:", round(ordinary_w, 3))
print("weighted IS:", round(weighted_w, 3))

plt.figure(figsize=(4.8, 3))
plt.bar(np.arange(len(weights_w)), weights_w, color="darkorange")
plt.title("5: importance weights can be uneven")
plt.xlabel("episode")
plt.ylabel("rho")
plt.show()

▶ What you'll see: one episode has zero support and one has a large weight, the source of off-policy variance.

*Why it's done this way:* off-policy data is biased toward the behavior policy unless we correct for sampling probabilities. The product ratio is exactly the probability of the trajectory under $\pi$ divided by its probability under $b$, but products can explode or vanish, which is why policy support is a major pitfall.


## ✍️ Toy Examples

> ✍️ **Toy examples — trace each mechanic by hand.** Separate from the walkthrough above, here
> is one tiny, fully hand-traceable toy per computational mechanic in this lesson. Each uses a few
> small numbers, prints the intermediate values with inline `# ->` comments, draws one picture, and
> ends with an `assert` that pins the answer. Run them top to bottom.

### ✍️ Toy 1 · Backward recursion makes suffix returns

Monte Carlo waits for an episode to finish, then computes one return for each time step by walking
backward through the rewards.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

t1_rng = np.random.default_rng(0)
t1_rewards = np.array([0.0, 2.0, 4.0])
t1_gamma = 0.5
t1_returns = np.zeros_like(t1_rewards)
t1_running = 0.0
for t1_t in range(t1_rewards.size - 1, -1, -1):
    t1_running = t1_rewards[t1_t] + t1_gamma * t1_running
    t1_returns[t1_t] = t1_running
    print("t", t1_t, "running return", round(float(t1_running), 3))

# -> t2:4.0, t1:4.0, t0:2.0
print("rewards:", t1_rewards.tolist())           # -> [0.0, 2.0, 4.0]
print("returns:", t1_returns.tolist())           # -> [2.0, 4.0, 4.0]

plt.figure(figsize=(4.3, 2.6))
plt.bar(["G0", "G1", "G2"], t1_returns, color="teal")
plt.title("Toy 1 · suffix returns")
plt.ylabel("return")
plt.show()
assert np.allclose(t1_returns, np.array([2.0, 4.0, 4.0]))

▶ What you'll see: later visits receive shorter reward ledgers than earlier visits.

### ✍️ Toy 2 · First-visit and every-visit count different samples

When a state repeats, first-visit Monte Carlo keeps only the first return from the episode; every-visit
Monte Carlo keeps them all.

In [ ]:
import numpy as np

t2_rng = np.random.default_rng(0)
t2_states = np.array([0, 1, 0, 2])
t2_rewards = np.array([0.0, 1.0, 2.0, 0.0])
t2_gamma = 0.5
t2_returns = np.zeros_like(t2_rewards)
t2_running = 0.0
for t2_t in range(t2_rewards.size - 1, -1, -1):
    t2_running = t2_rewards[t2_t] + t2_gamma * t2_running
    t2_returns[t2_t] = t2_running
t2_state0_returns = t2_returns[t2_states == 0]    # -> [1.0, 2.0]
t2_first = float(t2_state0_returns[0])            # -> 1.0
t2_every = float(np.mean(t2_state0_returns))      # -> 1.5

print("states:", t2_states.tolist())             # -> [0, 1, 0, 2]
print("returns:", t2_returns.tolist())           # -> [1.0, 2.0, 2.0, 0.0]
print("state-0 returns:", t2_state0_returns.tolist())  # -> [1.0, 2.0]
print("first-visit estimate:", t2_first)         # -> 1.0
print("every-visit estimate:", t2_every)         # -> 1.5

plt.figure(figsize=(4.3, 2.6))
plt.bar(["first", "every"], [t2_first, t2_every], color=["teal", "purple"])
plt.scatter([0, 1, 1], [t2_first, t2_state0_returns[0], t2_state0_returns[1]], color="black", zorder=3)
plt.title("Toy 2 · counting repeated visits")
plt.ylabel("state-0 estimate")
plt.show()
assert t2_first == 1.0 and t2_every == 1.5

▶ What you'll see: every-visit averages both state-0 returns, so it lands between `1.0` and `2.0`.

### ✍️ Toy 3 · Incremental averaging equals the ordinary mean

The running update gives each new return exactly one share of the sample average.

In [ ]:
import numpy as np

t3_rng = np.random.default_rng(0)
t3_samples = np.array([4.0, 2.0, 6.0, 8.0])
t3_mean = 0.0
t3_means = []
for t3_n, t3_sample in enumerate(t3_samples, start=1):
    t3_error = t3_sample - t3_mean
    t3_mean = t3_mean + t3_error / t3_n
    t3_means.append(t3_mean)
    print("n", t3_n, "sample", t3_sample, "mean", round(float(t3_mean), 3))
# -> means: 4.0, 3.0, 4.0, 5.0
t3_means = np.array(t3_means)
t3_ordinary = float(np.mean(t3_samples))          # -> 5.0

print("samples:", t3_samples.tolist())           # -> [4.0, 2.0, 6.0, 8.0]
print("running means:", t3_means.tolist())       # -> [4.0, 3.0, 4.0, 5.0]
print("ordinary mean:", t3_ordinary)             # -> 5.0

plt.figure(figsize=(4.4, 2.6))
plt.plot(np.arange(1, t3_samples.size + 1), t3_means, marker="o", color="seagreen")
plt.axhline(t3_ordinary, color="black", linestyle="--", label="ordinary mean")
plt.title("Toy 3 · running sample mean")
plt.xlabel("sample count")
plt.ylabel("estimate")
plt.legend()
plt.show()
assert t3_means[-1] == t3_ordinary

▶ What you'll see: the incremental estimate finishes at the same `5.0` as the direct mean.

### ✍️ Toy 4 · More episodes reduce mean-return noise

If return noise has standard deviation `σ`, the standard error of the average shrinks like
`σ / sqrt(n)`.

In [ ]:
import numpy as np

t4_rng = np.random.default_rng(0)
t4_episode_counts = np.array([1, 2, 4, 8])
t4_sigma = 2.0
t4_standard_error = t4_sigma / np.sqrt(t4_episode_counts)  # -> [2.0, 1.414, 1.0, 0.707]
t4_ratio = t4_standard_error[0] / t4_standard_error[-1]   # -> 2.828

print("episode counts:", t4_episode_counts.tolist())     # -> [1, 2, 4, 8]
print("standard errors:", np.round(t4_standard_error, 3).tolist())  # -> [2.0, 1.414, 1.0, 0.707]
print("noise shrink ratio:", round(float(t4_ratio), 3))   # -> 2.828

plt.figure(figsize=(4.3, 2.6))
plt.plot(t4_episode_counts, t4_standard_error, marker="o", color="purple")
plt.title("Toy 4 · averaging lowers variance")
plt.xlabel("episodes averaged")
plt.ylabel("standard error")
plt.show()
assert t4_standard_error[-1] < t4_standard_error[0]

▶ What you'll see: averaging eight episodes has much less noise than using one episode.

### ✍️ Toy 5 · MC control updates action values from returns

For control, the return is attached to a state-action pair, then `Q(s,a)` is averaged and greedified.

In [ ]:
import numpy as np

t5_rng = np.random.default_rng(0)
t5_episode = [(0, 1, 0.0), (1, 0, 3.0)]
t5_gamma = 0.5
t5_Q = np.zeros((2, 2))
t5_N = np.zeros((2, 2))
t5_G = 0.0
t5_returns = []
for t5_s, t5_a, t5_r in reversed(t5_episode):
    t5_G = t5_r + t5_gamma * t5_G
    t5_returns.append((t5_s, t5_a, t5_G))
t5_returns = list(reversed(t5_returns))
for t5_s, t5_a, t5_g in t5_returns:
    t5_N[t5_s, t5_a] = t5_N[t5_s, t5_a] + 1.0
    t5_Q[t5_s, t5_a] = t5_Q[t5_s, t5_a] + (t5_g - t5_Q[t5_s, t5_a]) / t5_N[t5_s, t5_a]
t5_policy = np.argmax(t5_Q, axis=1)              # -> [1, 0]

print("state-action returns:", [(s, a, round(g, 3)) for s, a, g in t5_returns])  # -> [(0,1,1.5), (1,0,3.0)]
print("Q table:", t5_Q.tolist())                 # -> [[0.0, 1.5], [3.0, 0.0]]
print("visit counts:", t5_N.tolist())            # -> [[0.0, 1.0], [1.0, 0.0]]
print("greedy policy:", t5_policy.tolist())      # -> [1, 0]

plt.figure(figsize=(4.0, 2.8))
plt.imshow(t5_Q, cmap="viridis", aspect="auto")
plt.colorbar(label="Q")
plt.title("Toy 5 · MC action-value table")
plt.xlabel("action")
plt.ylabel("state")
plt.show()
assert t5_policy.tolist() == [1, 0]

▶ What you'll see: only visited state-action cells change, and greedification follows the larger entries.

### ✍️ Toy 6 · Epsilon-greedy keeps action support

Exploration means every action keeps nonzero probability, while the greedy action receives the extra
mass.

In [ ]:
import numpy as np

t6_rng = np.random.default_rng(0)
t6_q = np.array([1.0, 3.0, 0.0])
t6_epsilon = 0.3
t6_n_actions = t6_q.size
t6_greedy = int(np.argmax(t6_q))                  # -> 1
t6_probs = np.ones(t6_n_actions) * (t6_epsilon / t6_n_actions)
t6_probs[t6_greedy] = t6_probs[t6_greedy] + (1.0 - t6_epsilon)
t6_expected = float(t6_probs @ t6_q)              # -> 2.5

print("Q row:", t6_q.tolist())                   # -> [1.0, 3.0, 0.0]
print("greedy action:", t6_greedy)               # -> 1
print("epsilon-greedy probs:", np.round(t6_probs, 3).tolist())  # -> [0.1, 0.8, 0.1]
print("expected Q under policy:", round(t6_expected, 3))         # -> 2.5

plt.figure(figsize=(4.3, 2.6))
plt.bar(["a0", "a1", "a2"], t6_probs, color="darkorange")
plt.ylim(0, 1)
plt.title("Toy 6 · ε-greedy support")
plt.ylabel("probability")
plt.show()
assert np.allclose(t6_probs, np.array([0.1, 0.8, 0.1]))

▶ What you'll see: even non-greedy actions keep probability `0.1`.

### ✍️ Toy 7 · Importance sampling corrects off-policy data

Off-policy evaluation multiplies the return by how much more or less likely the trajectory is under
the target policy than under the behavior policy.

In [ ]:
import numpy as np

t7_rng = np.random.default_rng(0)
t7_pi_probs = np.array([0.6, 0.9])
t7_b_probs = np.array([0.3, 0.6])
t7_ratios = t7_pi_probs / t7_b_probs              # -> [2.0, 1.5]
t7_rho = float(np.prod(t7_ratios))                # -> 3.0
t7_return = 2.0
t7_weighted_return = t7_rho * t7_return           # -> 6.0
t7_weights = np.array([3.0, 0.5])
t7_returns = np.array([2.0, 4.0])
t7_ordinary = float(np.mean(t7_weights * t7_returns))              # -> 4.0
t7_weighted = float(np.sum(t7_weights * t7_returns) / np.sum(t7_weights))  # -> 2.286

print("per-step ratios:", t7_ratios.tolist())    # -> [2.0, 1.5]
print("rho:", t7_rho)                             # -> 3.0
print("weighted return:", t7_weighted_return)    # -> 6.0
print("ordinary IS:", round(t7_ordinary, 3))     # -> 4.0
print("weighted IS:", round(t7_weighted, 3))     # -> 2.286

plt.figure(figsize=(4.3, 2.6))
plt.bar(["episode 0", "episode 1"], t7_weights, color="crimson")
plt.title("Toy 7 · importance weights")
plt.ylabel("ρ")
plt.show()
assert t7_rho == 3.0 and round(t7_weighted, 3) == 2.286

▶ What you'll see: the first episode counts much more because it is three times as likely under the target policy.

## 🛠️ Setup

In [ ]:
import numpy as np  # load NumPy for arrays, sampling, table updates, and assertions.
import matplotlib.pyplot as plt  # load Matplotlib for return curves, value plots, and policy diagnostics.
np.random.seed(0)  # make all random examples reproducible across notebook runs.

## 🟢 Basics (warm-up)

### Basic 1 — Compute one discounted return

**Goal.** Turn immediate rewards into one Monte Carlo return, because the learning target is discounted consequence rather than the first reward. We build it in 2 steps.

In [ ]:
rewards_b1 = np.array([1.0, 0.0, 2.0])  # store the reward stream after a state visit.
gamma_b1 = 0.9  # choose the discount factor used in the lesson grounding.

print("rewards_b1:", rewards_b1)  # inspect the immediate rewards.

▶ What you'll see: a delayed reward of `2` appears two steps after the first reward.

In [ ]:
discounts_b1 = gamma_b1 ** np.arange(len(rewards_b1))  # build [1, gamma, gamma^2].
G_b1 = float(np.sum(discounts_b1 * rewards_b1))  # compute the discounted return.

print("discounts_b1:", np.round(discounts_b1, 3), "G_b1:", round(G_b1, 3))  # inspect the calculation.

assert round(G_b1, 3) == 2.62  # verify 1 + 0.9*0 + 0.9^2*2.

▶ What you'll see: the delayed reward contributes `1.620`, so the return is `2.620`.

In [ ]:
contrib_b1 = discounts_b1 * rewards_b1  # discounted contribution from each reward.
plt.figure(figsize=(4.5, 3))
plt.bar(np.arange(len(rewards_b1)), contrib_b1, color="teal")
plt.axhline(G_b1, color="black", linestyle="--", label="total return")
plt.title("Basic 1: discounted reward contributions")
plt.xlabel("time step")
plt.ylabel("discounted reward")
plt.legend()
plt.show()

▶ What you'll see: the final delayed reward is discounted but still supplies most of the return.

👀 Takeaway: reward is immediate, while return is the discounted future ledger.

### Basic 2 — Compute returns for every time step

**Goal.** Build all suffix returns in one episode, because each visited state receives the return from its own time step onward. We build it in 2 steps.

In [ ]:
rewards_b2 = np.array([0.0, 1.0, 3.0])  # define rewards after each time step.
gamma_b2 = 0.5  # use a small discount so suffix differences are obvious.
returns_b2 = np.zeros_like(rewards_b2)  # allocate one return target per time step.

print("episode length:", len(rewards_b2))  # inspect how many targets we need.

▶ What you'll see: three rewards create three Monte Carlo targets.

In [ ]:
running_b2 = 0.0  # initialize the suffix accumulator after the terminal state.
for t_b2 in range(len(rewards_b2) - 1, -1, -1):  # move backward from the episode end.
    running_b2 = rewards_b2[t_b2] + gamma_b2 * running_b2  # one-step return recursion.
    returns_b2[t_b2] = running_b2  # save G_t.

print("returns_b2:", returns_b2)  # inspect [1.25, 2.5, 3.0].

assert np.allclose(returns_b2, [1.25, 2.5, 3.0])  # concrete suffix-return check.

▶ What you'll see: later returns are larger here because the big reward is near the end.

In [ ]:
plt.figure(figsize=(4.5, 3))
plt.plot(np.arange(len(returns_b2)), returns_b2, marker="o", color="seagreen")
plt.bar(np.arange(len(rewards_b2)), rewards_b2, alpha=0.25, color="gray", label="immediate reward")
plt.title("Basic 2: suffix returns by time step")
plt.xlabel("time step")
plt.ylabel("return")
plt.legend()
plt.show()

▶ What you'll see: each return includes a different suffix of the same reward sequence.

👀 Takeaway: backward recursion is the cleanest way to compute all returns from one finished episode.

### Basic 3 — Average returns into a value estimate

**Goal.** Estimate one state value from sampled returns, because Monte Carlo prediction is a sample average of realized outcomes. We build it in 2 steps.

In [ ]:
returns_b3 = np.array([2.0, 4.0, 1.0, 3.0])  # returns observed after visits to one state.
V_b3 = float(np.mean(returns_b3))  # ordinary Monte Carlo estimate.

print("returns_b3:", returns_b3)  # inspect evidence.
print("V_b3:", round(V_b3, 3))  # inspect the sample average.

assert round(V_b3, 3) == 2.5  # verify the average.

▶ What you'll see: the value estimate is the center of the sampled returns.

In [ ]:
plt.figure(figsize=(4, 3))  # create a compact return plot.
plt.scatter(np.arange(len(returns_b3)), returns_b3, color="teal")  # show each sampled return.
plt.axhline(V_b3, color="black", linestyle="--", label="mean V")  # show the estimate.
plt.title("Basic 3: value as average return")
plt.xlabel("visit")
plt.ylabel("return")
plt.legend()
plt.show()

▶ What you'll see: the dashed line is the Monte Carlo value estimate.

👀 Takeaway: $V(s)$ is estimated by averaging returns observed after visits to $s$.

### Basic 4 — Update a value incrementally

**Goal.** Reproduce the same average without storing all returns, because online Monte Carlo keeps counts and current estimates. We build it in 2 steps.

In [ ]:
samples_b4 = np.array([2.0, 4.0, 1.0, 3.0])  # same returns as Basic 3.
V_b4 = 0.0  # start with no value estimate.
N_b4 = 0  # start with no visits counted.

print("initial V_b4:", V_b4, "N_b4:", N_b4)  # inspect initial table state.

▶ What you'll see: the estimate starts at zero before any returns arrive.

In [ ]:
history_b4 = []  # track the estimate after each sample.
for G_b4 in samples_b4:  # process one return at a time.
    N_b4 += 1  # count this visit.
    V_b4 += (G_b4 - V_b4) / N_b4  # incremental sample mean.
    history_b4.append(V_b4)  # store for inspection.

print("history_b4:", np.round(history_b4, 3), "final:", round(V_b4, 3))  # inspect updates.

assert round(V_b4, 3) == 2.5  # same estimate as direct averaging.

▶ What you'll see: the final incremental estimate matches the ordinary mean.

In [ ]:
plt.figure(figsize=(4.5, 3))
plt.plot(np.arange(1, len(history_b4) + 1), history_b4, marker="o", color="purple")
plt.axhline(np.mean(samples_b4), color="black", linestyle="--", label="direct mean")
plt.title("Basic 4: incremental MC average")
plt.xlabel("number of returns")
plt.ylabel("V estimate")
plt.legend()
plt.show()

▶ What you'll see: the running estimate lands on the same value as the direct average.

👀 Takeaway: the update `V += (G - V)/N` is just a memory-efficient sample average.

### Basic 5 — Separate first-visit from every-visit counting

**Goal.** Count repeated states two ways, because first-visit and every-visit MC use different samples from the same episode. We build it in 2 steps.

In [ ]:
states_b5 = np.array([0, 1, 0, 2])  # state 0 appears twice in one episode.
returns_b5 = np.array([0.81, 0.9, 1.0, 0.0])  # returns after each visit.

print("states_b5:", states_b5)  # inspect repeated state visits.
print("returns_b5:", returns_b5)  # inspect targets attached to those visits.

▶ What you'll see: state `0` has two possible returns in the same episode.

In [ ]:
first_b5 = returns_b5[np.where(states_b5 == 0)[0][0]]  # first occurrence of state 0.
every_b5 = float(np.mean(returns_b5[states_b5 == 0]))  # average all state-0 occurrences.

print("first visit estimate:", round(float(first_b5), 3), "every visit estimate:", round(every_b5, 3))

assert round(every_b5, 3) == 0.905  # verify (0.81 + 1.0) / 2.

▶ What you'll see: every-visit differs because the second visit supplies another target.

In [ ]:
plt.figure(figsize=(4.5, 3))
plt.bar(["first", "every"], [first_b5, every_b5], color=["teal", "purple"])
plt.scatter([0, 1, 1], [first_b5, returns_b5[states_b5 == 0][0], returns_b5[states_b5 == 0][1]], color="black", zorder=3)
plt.title("Basic 5: first-visit vs every-visit")
plt.ylabel("state-0 estimate")
plt.show()

▶ What you'll see: every-visit averages both state-0 samples, while first-visit keeps only the first.

👀 Takeaway: first-visit counts one return per episode per state; every-visit counts every occurrence.

### Basic 6 — Store state values in a table

**Goal.** Update a small value table from one episode, because tabular Monte Carlo stores one estimate per state. We build it in 2 steps.

In [ ]:
states_b6 = np.array([0, 1, 2])  # states visited in one short episode.
returns_b6 = np.array([2.7, 3.0, 0.0])  # corresponding returns from those states.
V_b6 = np.zeros(3)  # initialize values for three states.
N_b6 = np.zeros(3)  # initialize visit counts for three states.

print("V shape:", V_b6.shape)  # inspect scalar value per state.

▶ What you'll see: `V` has shape `|S|`, not `|S|×|A|`.

In [ ]:
for s_b6, G_b6 in zip(states_b6, returns_b6):  # update each visited state.
    N_b6[s_b6] += 1  # count the visit.
    V_b6[s_b6] += (G_b6 - V_b6[s_b6]) / N_b6[s_b6]  # sample-average update.

print("V_b6:", np.round(V_b6, 3))  # inspect updated values.

assert np.allclose(V_b6, [2.7, 3.0, 0.0])  # verify one-sample averages.

▶ What you'll see: each visited state's value becomes its observed return.

In [ ]:
plt.figure(figsize=(4.5, 3))
plt.bar(np.arange(len(V_b6)), V_b6, color="seagreen")
plt.title("Basic 6: tabular state values")
plt.xlabel("state")
plt.ylabel("V(s)")
plt.show()

▶ What you'll see: the value table is one estimated return per state.

👀 Takeaway: a state-value table maps states to expected returns under the current policy.

### Basic 7 — Store action values in a table

**Goal.** Update $Q(s,a)$ rather than $V(s)$, because control needs values for actions inside each state. We build it in 2 steps.

In [ ]:
visits_b7 = [(0, 1, 2.43), (1, 0, 2.7), (2, 1, 3.0)]  # (state, action, return) samples.
Q_b7 = np.zeros((3, 2))  # three states by two actions.
N_b7 = np.zeros_like(Q_b7)  # matching counts for state-action visits.

print("Q shape:", Q_b7.shape)  # inspect |S| x |A| shape.

▶ What you'll see: action values require one column per action.

In [ ]:
for s_b7, a_b7, G_b7 in visits_b7:  # update visited state-action cells.
    N_b7[s_b7, a_b7] += 1  # count this state-action return.
    Q_b7[s_b7, a_b7] += (G_b7 - Q_b7[s_b7, a_b7]) / N_b7[s_b7, a_b7]  # average return.

print("Q_b7:\n", np.round(Q_b7, 3))  # inspect action-value table.

assert round(float(Q_b7[0, 1]), 3) == 2.43  # verify the updated cell.

▶ What you'll see: only the sampled actions have nonzero estimates.

In [ ]:
plt.figure(figsize=(4.5, 3))
plt.imshow(Q_b7, cmap="viridis", aspect="auto")
plt.colorbar(label="Q(s,a)")
plt.title("Basic 7: action-value table")
plt.xlabel("action")
plt.ylabel("state")
plt.show()

▶ What you'll see: action values occupy state-action cells instead of one scalar per state.

👀 Takeaway: $Q$ is the object control needs because it ranks actions, not just states.

### Basic 8 — Choose an epsilon-greedy action

**Goal.** Convert action values into a stochastic policy, because exploration prevents the agent from trusting early estimates too much. We build it in 2 steps.

In [ ]:
Qrow_b8 = np.array([1.0, 3.0, 2.0])  # action values in one state.
epsilon_b8 = 0.2  # exploration probability.
greedy_b8 = int(np.argmax(Qrow_b8))  # best estimated action.

print("greedy action:", greedy_b8)  # inspect exploitation choice.

▶ What you'll see: action `1` has the largest estimated return.

In [ ]:
probs_b8 = np.ones(3) * epsilon_b8 / 3  # exploration mass spread across all actions.
probs_b8[greedy_b8] += 1 - epsilon_b8  # exploitation mass added to the greedy action.

print("epsilon-greedy probabilities:", np.round(probs_b8, 3))  # inspect policy probabilities.

assert np.allclose(np.round(probs_b8, 3), [0.067, 0.867, 0.067])  # verify probability mass.

▶ What you'll see: the greedy action is likely, but every action keeps support.

In [ ]:
plt.figure(figsize=(4.5, 3))
plt.bar(np.arange(len(probs_b8)), probs_b8, color=["gray" if a_b8 != greedy_b8 else "seagreen" for a_b8 in range(len(probs_b8))])
plt.title("Basic 8: epsilon-greedy action support")
plt.xlabel("action")
plt.ylabel("selection probability")
plt.ylim(0, 1)
plt.show()

▶ What you'll see: the greedy action dominates, but exploration keeps nonzero probability everywhere.

👀 Takeaway: epsilon-greedy makes improvement safe enough to keep discovering better actions.

### Basic 9 — Compute a softmax policy

**Goal.** Turn logits into action probabilities, because policy weighting connects preferences to expected consequences. We build it in 2 steps.

In [ ]:
logits_b9 = np.array([1.0, 0.0])  # unnormalized action preferences.
exp_b9 = np.exp(logits_b9 - np.max(logits_b9))  # stable exponentials.

print("exp logits:", np.round(exp_b9, 3))  # inspect relative weights.

▶ What you'll see: subtracting the maximum keeps the same probabilities but avoids large exponentials.

In [ ]:
probs_b9 = exp_b9 / np.sum(exp_b9)  # normalize to probabilities.
expected_reward_b9 = float(probs_b9 @ np.array([2.0, 0.0]))  # expected immediate reward under policy.

print("softmax probabilities:", np.round(probs_b9, 3), "expected reward:", round(expected_reward_b9, 3))

assert np.allclose(np.round(probs_b9, 3), [0.731, 0.269])  # verify lesson grounding.

▶ What you'll see: action 0 gets probability `0.731`, yielding expected reward `1.462`.

In [ ]:
plt.figure(figsize=(4.5, 3))
plt.bar(np.arange(len(probs_b9)), probs_b9, color="teal")
plt.title("Basic 9: softmax policy probabilities")
plt.xlabel("action")
plt.ylabel("probability")
plt.ylim(0, 1)
plt.show()

▶ What you'll see: the higher logit gets more probability, but the lower-logit action remains possible.

👀 Takeaway: policy probabilities weight outcomes, so changing logits changes expected consequence.

### Basic 10 — Compute one importance weight

**Goal.** Reweight an off-policy episode, because behavior-policy data must be corrected before estimating a target policy. We build it in 2 steps.

In [ ]:
pi_b10 = np.array([0.8, 0.2])  # target-policy probabilities for sampled actions.
b_b10 = np.array([0.5, 0.5])  # behavior-policy probabilities for the same actions.
ratios_b10 = pi_b10 / b_b10  # per-decision probability ratios.

print("ratios_b10:", ratios_b10)  # inspect likelihood correction at each step.

▶ What you'll see: the first sampled action is up-weighted and the second is down-weighted.

In [ ]:
rho_b10 = float(np.prod(ratios_b10))  # multiply ratios over the trajectory.
G_b10 = 3.0  # observed return under the behavior policy.

print("rho:", round(rho_b10, 3), "rho*G:", round(rho_b10 * G_b10, 3))  # inspect weighted contribution.

assert round(rho_b10, 3) == 0.64  # verify the trajectory weight.

▶ What you'll see: the sampled return contributes `1.920` to an ordinary IS average.

In [ ]:
plt.figure(figsize=(4.5, 3))
plt.bar(np.arange(len(ratios_b10)), ratios_b10, color="darkorange")
plt.axhline(rho_b10, color="black", linestyle="--", label="product rho")
plt.title("Basic 10: per-step ratios multiply")
plt.xlabel("decision step")
plt.ylabel("importance ratio")
plt.legend()
plt.show()

▶ What you'll see: the trajectory weight is the product of an up-weight and a down-weight.

👀 Takeaway: importance sampling corrects policy mismatch, but products of ratios can be noisy.

## 🟡 Easy

### Easy 1 — First-visit MC prediction on a tiny random walk

**Goal.** Estimate state values from many complete episodes, because Monte Carlo prediction learns by averaging realized returns under a fixed policy. We build it in 4 steps.

In [ ]:
n_states_e1 = 5  # states 0 and 4 are terminal; 1,2,3 are nonterminal.
gamma_e1 = 1.0  # undiscounted episodic return for the random walk.
V_e1 = np.zeros(n_states_e1)  # value estimates for all states.
N_e1 = np.zeros(n_states_e1)  # first-visit counts.

print("nonterminal states:", [1, 2, 3])  # inspect states that will be updated.

▶ What you'll see: only the middle states receive Monte Carlo prediction updates.

In [ ]:
def episode_e1(rng_e1):  # sample one random-walk episode.
    s_e1 = 2  # start in the center.
    states_e1, rewards_e1 = [], []  # store trajectory pieces.
    while s_e1 not in (0, 4):  # continue until a terminal state.
        states_e1.append(s_e1)  # record current state.
        step_e1 = -1 if rng_e1.random() < 0.5 else 1  # random left/right action.
        s_e1 = s_e1 + step_e1  # move to next state.
        rewards_e1.append(1.0 if s_e1 == 4 else 0.0)  # reward only at the right terminal.
    return states_e1, rewards_e1  # return complete episode.
rng_e1 = np.random.default_rng(1)  # reproducible episodes.

print("sample episode:", episode_e1(rng_e1))  # inspect one trajectory.

▶ What you'll see: a complete state list and reward list ending at a terminal state.

In [ ]:
rng_e1 = np.random.default_rng(1)  # reset so the training run is reproducible.
for ep_e1 in range(500):  # collect many full episodes.
    states_run_e1, rewards_run_e1 = episode_e1(rng_e1)  # sample one episode.
    G_e1 = 0.0  # terminal suffix return.
    returns_run_e1 = []  # returns for each time step.
    for r_e1 in reversed(rewards_run_e1):  # compute returns backward.
        G_e1 = r_e1 + gamma_e1 * G_e1  # discounted suffix return.
        returns_run_e1.append(G_e1)  # save return.
    returns_run_e1 = list(reversed(returns_run_e1))  # align with states.
    seen_e1 = set()  # first-visit filter for this episode.
    for s_e1, Gt_e1 in zip(states_run_e1, returns_run_e1):  # scan visits.
        if s_e1 not in seen_e1:  # first visit only.
            N_e1[s_e1] += 1  # count state sample.
            V_e1[s_e1] += (Gt_e1 - V_e1[s_e1]) / N_e1[s_e1]  # update mean.
            seen_e1.add(s_e1)  # prevent another update in same episode.

print("V_e1:", np.round(V_e1, 3))  # inspect learned values.

assert V_e1[1] < V_e1[2] < V_e1[3]  # values should rise toward the rewarding terminal.

▶ What you'll see: state values increase from left to right because right terminal pays reward.

In [ ]:
plt.figure(figsize=(5, 3))
plt.bar(np.arange(n_states_e1), V_e1, color="seagreen")
plt.title("Easy 1: first-visit MC values")
plt.xlabel("state")
plt.ylabel("estimated V")
plt.show()

▶ What you'll see: terminal states stay at zero here; nonterminal values form an increasing ramp.

👀 Takeaway: first-visit MC prediction estimates expected return by one update per state per episode.

### Easy 2 — Every-visit MC prediction on the same walk

**Goal.** Use every repeated state visit, because every-visit MC treats each occurrence as a return sample. We build it in 4 steps.

In [ ]:
n_states_e2 = 5  # same random-walk states.
gamma_e2 = 1.0  # undiscounted return.
V_e2 = np.zeros(n_states_e2)  # every-visit value estimates.
N_e2 = np.zeros(n_states_e2)  # every-visit counts.

print("initial counts:", N_e2)  # inspect empty visit counters.

▶ What you'll see: all state counts start at zero.

In [ ]:
def episode_e2(rng_e2):  # sample one episode.
    s_e2 = 2  # center start.
    states_e2, rewards_e2 = [], []  # trajectory containers.
    while s_e2 not in (0, 4):  # stop at terminals.
        states_e2.append(s_e2)  # record current state.
        s_e2 += -1 if rng_e2.random() < 0.5 else 1  # move left or right.
        rewards_e2.append(1.0 if s_e2 == 4 else 0.0)  # reward on right terminal.
    return states_e2, rewards_e2  # complete episode.
rng_e2 = np.random.default_rng(2)  # reproducible sampler.

print("sample length:", len(episode_e2(rng_e2)[0]))  # inspect one episode length.

▶ What you'll see: random walks can revisit states and have variable length.

In [ ]:
rng_e2 = np.random.default_rng(2)  # reset sampler.
for ep_e2 in range(500):  # many episodes.
    states_run_e2, rewards_run_e2 = episode_e2(rng_e2)  # sample trajectory.
    G_e2 = 0.0  # suffix return.
    returns_run_e2 = []  # per-time returns.
    for r_e2 in reversed(rewards_run_e2):  # backward pass.
        G_e2 = r_e2 + gamma_e2 * G_e2  # update suffix return.
        returns_run_e2.append(G_e2)  # collect.
    returns_run_e2 = list(reversed(returns_run_e2))  # align with visits.
    for s_e2, Gt_e2 in zip(states_run_e2, returns_run_e2):  # every visit updates.
        N_e2[s_e2] += 1  # count this occurrence.
        V_e2[s_e2] += (Gt_e2 - V_e2[s_e2]) / N_e2[s_e2]  # update mean.

print("counts_e2:", N_e2.astype(int), "V_e2:", np.round(V_e2, 3))  # inspect counts and values.

assert N_e2[2] > 500  # every-visit counts can exceed episode count because state 2 repeats.

▶ What you'll see: the center state can have more visits than the number of episodes.

In [ ]:
plt.figure(figsize=(5, 3))
plt.bar(np.arange(n_states_e2), N_e2, color="purple")
plt.title("Easy 2: every-visit counts")
plt.xlabel("state")
plt.ylabel("number of counted visits")
plt.show()

▶ What you'll see: frequently revisited states collect many more return samples.

👀 Takeaway: every-visit MC can use more samples per episode, but those samples are correlated.

### Easy 3 — Monte Carlo action-value control with epsilon-greedy improvement

**Goal.** Learn which first action reaches a reward more often, because MC control alternates return averaging and policy improvement. We build it in 4 steps.

In [ ]:
Q_e3 = np.zeros((2, 2))  # two states and two actions.
N_e3 = np.zeros_like(Q_e3)  # state-action counts.
epsilon_e3 = 0.2  # exploration probability.
gamma_e3 = 1.0  # undiscounted terminal reward.

print("Q_e3 shape:", Q_e3.shape)  # inspect action-value table.

▶ What you'll see: control uses a row of action values for each decision state.

In [ ]:
def choose_e3(Qrow_e3, rng_e3):  # epsilon-greedy action selector.
    if rng_e3.random() < epsilon_e3:  # exploration branch.
        return int(rng_e3.integers(2))  # choose either action uniformly.
    return int(np.argmax(Qrow_e3))  # exploitation branch.
def rollout_e3(rng_e3):  # one tiny episode.
    a0_e3 = choose_e3(Q_e3[0], rng_e3)  # choose action in state 0.
    if a0_e3 == 1:  # action 1 jumps to rewarding terminal.
        return [(0, a0_e3, 1.0)]  # immediate reward.
    a1_e3 = choose_e3(Q_e3[1], rng_e3)  # otherwise visit state 1.
    reward_e3 = 1.0 if a1_e3 == 1 else 0.0  # state 1 action 1 succeeds.
    return [(0, a0_e3, 0.0), (1, a1_e3, reward_e3)]  # full episode.
rng_e3 = np.random.default_rng(3)  # reproducible control run.

print("sample rollout:", rollout_e3(rng_e3))  # inspect an episode format.

▶ What you'll see: each tuple contains `(state, action, reward-after-action)`.

In [ ]:
rng_e3 = np.random.default_rng(3)  # reset after inspection.
for ep_e3 in range(800):  # train from complete episodes.
    ep_run_e3 = rollout_e3(rng_e3)  # collect one episode using current Q.
    G_e3 = 0.0  # suffix return.
    visited_e3 = set()  # first-visit state-action filter.
    for s_e3, a_e3, r_e3 in reversed(ep_run_e3):  # walk backward.
        G_e3 = r_e3 + gamma_e3 * G_e3  # state-action return.
        key_e3 = (s_e3, a_e3)  # state-action identity.
        if key_e3 not in visited_e3:  # first visit to this pair in episode.
            N_e3[s_e3, a_e3] += 1  # count sample.
            Q_e3[s_e3, a_e3] += (G_e3 - Q_e3[s_e3, a_e3]) / N_e3[s_e3, a_e3]  # average.
            visited_e3.add(key_e3)  # avoid duplicate update.
policy_e3 = np.argmax(Q_e3, axis=1)  # greedy policy after learning.

print("Q_e3:\n", np.round(Q_e3, 3), "policy:", policy_e3)  # inspect learned control table.

assert policy_e3[0] == 1  # direct rewarding action should be preferred.

▶ What you'll see: action `1` in state `0` gets the largest value.

In [ ]:
plt.figure(figsize=(4.5, 3))
plt.imshow(Q_e3, cmap="viridis", aspect="auto")
plt.colorbar(label="Q")
plt.title("Easy 3: MC control Q table")
plt.xlabel("action")
plt.ylabel("state")
plt.show()

▶ What you'll see: brighter cells identify the greedy improved policy.

👀 Takeaway: MC control learns action values from complete returns, then improves by acting greedily with exploration.

### Easy 4 — Off-policy value estimation with weighted importance sampling

**Goal.** Estimate a target policy from behavior-policy episodes, because logged data often comes from a different policy than the one we want to evaluate. We build it in 3 steps.

In [ ]:
returns_e4 = np.array([1.0, 0.0, 1.0, 1.0, 0.0])  # observed episode returns under behavior.
weights_e4 = np.array([1.6, 0.0, 0.8, 1.2, 0.0])  # likelihood ratios for the target policy.

print("returns_e4:", returns_e4)  # inspect behavior-policy outcomes.
print("weights_e4:", weights_e4)  # inspect target/behavior corrections.

▶ What you'll see: some episodes have zero weight because the target policy would not take those actions.

In [ ]:
ordinary_e4 = float(np.mean(weights_e4 * returns_e4))  # ordinary importance sampling.
weighted_e4 = float(np.sum(weights_e4 * returns_e4) / np.sum(weights_e4))  # normalized weighted IS.

print("ordinary IS:", round(ordinary_e4, 3), "weighted IS:", round(weighted_e4, 3))  # inspect both estimates.

assert round(weighted_e4, 3) == 1.0  # all nonzero-weight episodes here have return 1.

▶ What you'll see: weighted IS normalizes by total support weight rather than episode count.

In [ ]:
plt.figure(figsize=(4.5, 3))
plt.bar(np.arange(len(weights_e4)), weights_e4 * returns_e4, color="darkorange")
plt.title("Easy 4: weighted off-policy contributions")
plt.xlabel("episode")
plt.ylabel("rho × return")
plt.show()

▶ What you'll see: only episodes compatible with the target policy contribute to the estimate.

👀 Takeaway: importance sampling fixes policy mismatch only where the behavior policy has target-policy support.

### Easy 5 — Visualize return variance shrinking with episodes

**Goal.** Show why many episodes are needed, because complete returns can be noisy even when they are unbiased. We build it in 3 steps.

In [ ]:
rng_e5 = np.random.default_rng(5)  # reproducible noisy returns.
true_value_e5 = 2.0  # the mean return we are trying to estimate.
returns_e5 = rng_e5.normal(loc=true_value_e5, scale=2.5, size=300)  # noisy complete returns.

print("first five returns:", np.round(returns_e5[:5], 3))  # inspect sample noise.

▶ What you'll see: individual returns can be far from the true value.

In [ ]:
running_e5 = np.cumsum(returns_e5) / np.arange(1, len(returns_e5) + 1)  # sample mean after each episode.

print("estimate after 1 episode:", round(float(running_e5[0]), 3))
print("estimate after 300 episodes:", round(float(running_e5[-1]), 3))

assert abs(running_e5[-1] - true_value_e5) < 0.35  # with this seed the average settles near truth.

▶ What you'll see: the early estimate is noisy; the late estimate is much closer to `2.0`.

In [ ]:
plt.figure(figsize=(5, 3))
plt.plot(running_e5, color="teal")
plt.axhline(true_value_e5, color="black", linestyle="--", label="true value")
plt.title("Easy 5: MC variance shrinks by averaging")
plt.xlabel("episode")
plt.ylabel("running estimate")
plt.legend()
plt.show()

▶ What you'll see: the running estimate wobbles before settling near the dashed true value.

👀 Takeaway: Monte Carlo targets avoid bootstrap bias, but the price is high variance from whole futures.

## 🔴 Advanced

### Advanced 1 — Compare first-visit and every-visit convergence

**Goal.** Run both estimators side by side, because they use the same episodes but count repeated visits differently. We build it in 4 steps.

In [ ]:
def episode_a1(rng_a1):  # random walk with repeated states possible.
    s_a1 = 2  # center start.
    states_a1, rewards_a1 = [], []  # trajectory storage.
    while s_a1 not in (0, 4):  # terminals at both ends.
        states_a1.append(s_a1)  # record state.
        s_a1 += -1 if rng_a1.random() < 0.5 else 1  # random movement.
        rewards_a1.append(1.0 if s_a1 == 4 else 0.0)  # reward at right terminal only.
    return states_a1, rewards_a1  # complete episode.
V_first_a1 = np.zeros(5); V_every_a1 = np.zeros(5)  # two value estimators.
N_first_a1 = np.zeros(5); N_every_a1 = np.zeros(5)  # matching counts.

print("estimators initialized")  # confirm setup.

▶ What you'll see: both methods start from identical zero tables.

In [ ]:
rng_a1 = np.random.default_rng(11)  # reproducible episodes.
trace_first_a1, trace_every_a1 = [], []  # track state-2 estimates.
for ep_a1 in range(1000):  # many complete episodes.
    states_run_a1, rewards_run_a1 = episode_a1(rng_a1)  # sample trajectory.
    G_a1 = 0.0; returns_run_a1 = []  # suffix return storage.
    for r_a1 in reversed(rewards_run_a1):  # backward return calculation.
        G_a1 = r_a1 + G_a1  # gamma=1.
        returns_run_a1.append(G_a1)  # save.
    returns_run_a1 = list(reversed(returns_run_a1))  # align.
    seen_a1 = set()  # first-visit state filter.
    for s_a1, Gt_a1 in zip(states_run_a1, returns_run_a1):  # process visits.
        N_every_a1[s_a1] += 1; V_every_a1[s_a1] += (Gt_a1 - V_every_a1[s_a1]) / N_every_a1[s_a1]  # every visit.
        if s_a1 not in seen_a1:  # first visit only once per episode.
            N_first_a1[s_a1] += 1; V_first_a1[s_a1] += (Gt_a1 - V_first_a1[s_a1]) / N_first_a1[s_a1]  # first visit.
            seen_a1.add(s_a1)  # mark counted.
    trace_first_a1.append(V_first_a1[2]); trace_every_a1.append(V_every_a1[2])  # track center value.

print("state-2 first/every:", round(float(V_first_a1[2]), 3), round(float(V_every_a1[2]), 3))

assert abs(V_first_a1[2] - 0.5) < 0.08 and abs(V_every_a1[2] - 0.5) < 0.08  # both estimate the center value.

▶ What you'll see: both estimates land near `0.5` for the symmetric center state.

In [ ]:
print("counts for state 2:", int(N_first_a1[2]), int(N_every_a1[2]))  # inspect different sample counts.

plt.figure(figsize=(5, 3))
plt.plot(trace_first_a1, label="first-visit", color="teal")
plt.plot(trace_every_a1, label="every-visit", color="purple", alpha=0.8)
plt.axhline(0.5, color="black", linestyle="--", label="true center value")
plt.title("Advanced 1: first vs every visit")
plt.xlabel("episode")
plt.ylabel("V(2)")
plt.legend()
plt.show()

▶ What you'll see: the curves converge to similar values but with different counting behavior.

👀 Takeaway: first-visit and every-visit MC share the same return target but differ in sample accounting.

### Advanced 2 — Ordinary versus weighted importance sampling variance

**Goal.** Compare off-policy estimators over many runs, because importance weights can make ordinary estimates extremely variable. We build it in 4 steps.

In [ ]:
rng_a2 = np.random.default_rng(22)  # reproducible simulation.
n_runs_a2 = 400  # number of repeated experiments.
n_eps_a2 = 30  # behavior episodes per experiment.
ordinary_est_a2 = []  # store ordinary IS estimates.
weighted_est_a2 = []  # store weighted IS estimates.

print("runs:", n_runs_a2, "episodes per run:", n_eps_a2)  # inspect simulation size.

▶ What you'll see: we will compare many small off-policy datasets.

In [ ]:
for run_a2 in range(n_runs_a2):  # repeat datasets.
    actions_a2 = rng_a2.random(n_eps_a2) < 0.5  # behavior chooses target-compatible action half the time.
    rewards_a2 = actions_a2.astype(float)  # reward is 1 only for compatible action.
    rho_a2 = np.where(actions_a2, 2.0, 0.0)  # target always chooses compatible action; behavior prob is 0.5.
    ordinary_est_a2.append(float(np.mean(rho_a2 * rewards_a2)))  # ordinary IS.
    weighted_est_a2.append(float(np.sum(rho_a2 * rewards_a2) / np.sum(rho_a2)) if np.sum(rho_a2) > 0 else 0.0)  # weighted IS.
ordinary_est_a2 = np.array(ordinary_est_a2); weighted_est_a2 = np.array(weighted_est_a2)  # arrays for summaries.

print("ordinary mean/std:", round(float(np.mean(ordinary_est_a2)), 3), round(float(np.std(ordinary_est_a2)), 3))
print("weighted mean/std:", round(float(np.mean(weighted_est_a2)), 3), round(float(np.std(weighted_est_a2)), 3))

assert np.std(weighted_est_a2) < np.std(ordinary_est_a2)  # weighted IS is steadier in this simple setup.

▶ What you'll see: both estimate near `1`, but ordinary IS has larger spread.

In [ ]:
plt.figure(figsize=(5, 3))
plt.hist(ordinary_est_a2, bins=20, alpha=0.6, label="ordinary IS", color="red")
plt.hist(weighted_est_a2, bins=20, alpha=0.6, label="weighted IS", color="teal")
plt.title("Advanced 2: IS estimator spread")
plt.xlabel("estimated value")
plt.ylabel("count")
plt.legend()
plt.show()

▶ What you'll see: weighted IS is concentrated near 1, while ordinary IS spreads across more values.

👀 Takeaway: importance sampling fixes bias from policy mismatch, but weighted normalization often reduces variance.

### Advanced 3 — Exploring starts for Monte Carlo control

**Goal.** Force all state-action pairs to be sampled, because greedy improvement cannot improve actions that never appear in episodes. We build it in 4 steps.

In [ ]:
Q_a3 = np.zeros((3, 2))  # three states, two actions.
N_a3 = np.zeros_like(Q_a3)  # state-action counts.
rng_a3 = np.random.default_rng(33)  # reproducible exploring starts.

print("initial Q_a3:\n", Q_a3)  # inspect empty table.

▶ What you'll see: no action has evidence before exploring starts.

In [ ]:
def finish_a3(s_a3, a_a3):  # deterministic toy consequence after chosen start.
    if s_a3 == 0 and a_a3 == 1:  # best immediate action.
        return [(s_a3, a_a3, 2.0)]  # high reward.
    if s_a3 == 1 and a_a3 == 1:  # good action in state 1.
        return [(s_a3, a_a3, 1.0)]  # moderate reward.
    return [(s_a3, a_a3, 0.0), (2, 1, 0.5)]  # otherwise drift to a small terminal payoff.

print("example start rollout:", finish_a3(0, 0))  # inspect trajectory format.

▶ What you'll see: starting from a weak action still creates a complete episode return.

In [ ]:
for ep_a3 in range(300):  # train with random starting state-action pairs.
    s0_a3 = int(rng_a3.integers(3))  # exploring start state.
    a0_a3 = int(rng_a3.integers(2))  # exploring start action.
    ep_run_a3 = finish_a3(s0_a3, a0_a3)  # complete trajectory from that start.
    G_a3 = 0.0  # suffix return.
    seen_a3 = set()  # first-visit state-action filter.
    for s_a3, a_a3, r_a3 in reversed(ep_run_a3):  # process complete episode.
        G_a3 = r_a3 + G_a3  # gamma=1.
        if (s_a3, a_a3) not in seen_a3:  # first visit to pair.
            N_a3[s_a3, a_a3] += 1  # count sample.
            Q_a3[s_a3, a_a3] += (G_a3 - Q_a3[s_a3, a_a3]) / N_a3[s_a3, a_a3]  # update mean.
            seen_a3.add((s_a3, a_a3))  # mark counted.

print("counts_a3:\n", N_a3.astype(int))  # inspect coverage.
print("Q_a3:\n", np.round(Q_a3, 3))  # inspect action values.

assert np.all(N_a3 > 0)  # exploring starts covered every pair.

▶ What you'll see: every state-action cell has at least one sample.

In [ ]:
plt.figure(figsize=(4.5, 3))
plt.imshow(Q_a3, cmap="viridis", aspect="auto")
plt.colorbar(label="Q")
plt.title("Advanced 3: exploring starts cover Q")
plt.xlabel("action")
plt.ylabel("state")
plt.show()

▶ What you'll see: all cells are learned because all starts were possible.

👀 Takeaway: exploring starts are a clean theoretical way to guarantee support for MC control.

### Advanced 4 — Compare Monte Carlo and one-step bootstrap targets

**Goal.** Contrast complete-return targets with a one-step target, because the target choice controls bias and variance. We build it in 3 steps.

In [ ]:
rewards_a4 = np.array([1.0, 0.0, 2.0])  # full future reward stream.
gamma_a4 = 0.9  # discount.
V_next_a4 = 0.8  # current estimate for the next state in a bootstrap method.

print("rewards_a4:", rewards_a4, "V_next:", V_next_a4)  # inspect target ingredients.

▶ What you'll see: MC has access to the whole episode; bootstrap uses a current estimate.

In [ ]:
mc_target_a4 = float(np.sum((gamma_a4 ** np.arange(len(rewards_a4))) * rewards_a4))  # complete return.
td_target_a4 = float(rewards_a4[0] + gamma_a4 * V_next_a4)  # one-step bootstrap target.

print("MC target:", round(mc_target_a4, 3), "one-step target:", round(td_target_a4, 3))

assert round(mc_target_a4, 3) == 2.62 and round(td_target_a4, 3) == 1.72  # grounded lesson numbers.

▶ What you'll see: the complete return is `2.620`, while the one-step target is `1.720`.

In [ ]:
plt.figure(figsize=(4, 3))
plt.bar(["MC complete G", "one-step r+γV"], [mc_target_a4, td_target_a4], color=["teal", "orange"])
plt.title("Advanced 4: target choice changes update")
plt.ylabel("target value")
plt.xticks(rotation=10)
plt.show()

▶ What you'll see: MC waits and uses the realized delayed reward; the bootstrap target stops after one reward plus an estimate.

👀 Takeaway: MC avoids bootstrapping bias by waiting, but waiting exposes it to full-episode variance.

### Advanced 5 — Add an exploration bonus to action selection

**Goal.** Compute a UCB-style exploration index, because uncertainty can be valuable even when the current mean is lower. We build it in 3 steps.

In [ ]:
means_a5 = np.array([0.55, 0.70, 0.40])  # current mean returns for three actions.
counts_a5 = np.array([5.0, 20.0, 2.0])  # how often each action has been sampled.
t_a5 = 20.0  # total decision time from the grounding example.
c_a5 = 1.0  # exploration strength.

print("means:", means_a5, "counts:", counts_a5)  # inspect exploitation and uncertainty inputs.

▶ What you'll see: action 2 has a low mean but very few samples.

In [ ]:
bonus_a5 = c_a5 * np.sqrt(2 * np.log(t_a5) / counts_a5)  # UCB uncertainty bonus.
ucb_a5 = means_a5 + bonus_a5  # optimistic action index.

print("bonus:", np.round(bonus_a5, 3))  # inspect uncertainty payments.
print("UCB:", np.round(ucb_a5, 3))  # inspect final action scores.

assert round(float(0.55 + np.sqrt(2 * np.log(20) / 5)), 3) == 1.645  # grounded bonus example.

▶ What you'll see: less-sampled actions receive larger bonuses.

In [ ]:
best_a5 = int(np.argmax(ucb_a5))  # choose the optimistic action.
plt.figure(figsize=(5, 3))
plt.bar(np.arange(3), means_a5, label="mean", color="gray")
plt.bar(np.arange(3), bonus_a5, bottom=means_a5, label="bonus", color="skyblue")
plt.title(f"Advanced 5: UCB selects action {best_a5}")
plt.xlabel("action")
plt.ylabel("mean + bonus")
plt.legend()
plt.show()

▶ What you'll see: the stacked bars show which action wins after adding exploration pressure.

👀 Takeaway: exploration bonuses deliberately pay for information, which helps MC control avoid premature greed.